In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging

logging.getLogger("pint").setLevel(logging.ERROR)

import pickle

import jax
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm

from gallifrey.inference import predictive_distribution, calculate_whitened_residuals
from gallifrey.util import example_lightcurve, example_transit_model
from gallifrey.visualisation import plot_masks, plot_prediction, plot_residuals

In [ ]:
rng_key = jax.random.PRNGKey(42)

plt.style.use("../figures/gpjax.mplstyle")
cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

## TRANSIT PARAMETER POSTERIOR (WHITE NOISE)

In [ ]:
models = {
    name: np.load(
        f"../data/processed/toy_data/mcmc_chains/{model}_parameter.npy",
    )
    for name, model in zip(
        ["Learned", "Periodic", "RBF"],
        ["gpmodel", "gpmodel_periodic_kernel", "gpmodel_rbf_kernel"],
    )
}
ground_truth = [0.1, 0.1, 0.3]

In [ ]:
dfs = []
names = ["Learned", "Periodic", "RBF"]
for name, model in models.items():
    df = pd.DataFrame(model, columns=["radius ratio", "$u_1$", "$u_2$"])
    df["Kernel"] = name
    dfs.append(df)
df = pd.concat(dfs)

In [ ]:
plot = sns.pairplot(
    df,
    corner=True,
    hue="Kernel",
    hue_order=["RBF", "Learned"],
    diag_kind="hist",
    kind="kde",
    diag_kws={"element": "step"},
    plot_kws={"fill": True, "alpha": 0.6, "levels": 4},
)

# Iterate over the axes
for i in range(3):
    for j in range(3):
        if plot.axes[i, j]:
            plot.axes[i, j].axvline(ground_truth[j], color="k")
            if i != j:
                plot.axes[i, j].scatter(ground_truth[j], ground_truth[i], color="k")
                plot.axes[i, j].axhline(ground_truth[i], color="k")

plt.savefig("../figures/004_plots/toy_example_corner.pdf")
plt.show()

## AR NOISE

In [ ]:
example = example_lightcurve(correlated=True)

In [ ]:
lc_parameter = np.load(
    "../data/processed/toy_data/mcmc_chains/gpmodel_ar_noise_parameter.npy"
)
gp_parameter = np.load(
    "../data/processed/toy_data/mcmc_chains/gpmodel_ar_noise_parameter_gp.npy"
)
with open("../data/processed/toy_data/gp_models/gpmodel_ar_noise", "rb") as file:
    gpmodel_ar = pickle.load(file)

In [ ]:
df = pd.DataFrame(lc_parameter, columns=["radius ratio", "$u_1$", "$u_2$"])
df.describe()

### Model Prediction

In [ ]:
rng_key, sample_key = jax.random.split(rng_key)
parameter_sample = lambda key, num: jax.random.choice(
    key, len(gp_parameter), shape=(num,)
)

samples_with_transit = []
samples_without_transit = []
for idx in tqdm(parameter_sample(sample_key, 500)):
    rng_key, sample_key = jax.random.split(rng_key)
    dist = predictive_distribution(
        gpmodel_ar,
        example["t"],
        x=example["t_sample"][example["transit_mask_sample"]],
        y=example["lc_sample"][example["transit_mask_sample"]],
        transit_model=example_transit_model,
        transit_parameter=lc_parameter[idx],
        gp_parameter=gp_parameter[idx],
    )
    sample = dist.sample(seed=sample_key)

    samples_with_transit.append(sample)
    samples_without_transit.append(
        sample - example_transit_model(example["t"], lc_parameter[idx])
    )
    del dist

In [ ]:
fig, ax = plt.subplots()
ax.scatter(
    example["t_sample"],
    example["lc_sample"],
    label="Data",
    color=cols[0],
    alpha=0.5,
)

ax = plot_prediction(
    ax,
    example["t"],
    jax.numpy.array(samples_with_transit),
    errorbar=68,
)
ax = plot_prediction(
    ax,
    example["t"][~example["transit_mask"]],
    jax.numpy.array(samples_without_transit)[:, ~example["transit_mask"]],
    kws_mean={"color": "grey", "label": "GP Background", "alpha": 0.7},
)
ax = plot_masks(ax, example["t"], example["transit_mask"])


ax.legend(loc="center left", bbox_to_anchor=(0, 0.23))
ax.set_ylabel("Flux")
ax.set_xlabel("Time")
plt.savefig("../figures/004_plots/model_distribution_ar_noise.pdf")

### Residuals

In [ ]:
rng_key, sample_key = jax.random.split(rng_key)
residuals = []
for idx in tqdm(parameter_sample(sample_key, 2500)):
    rng_key, sample_key = jax.random.split(rng_key)
    dist = predictive_distribution(
        gpmodel_ar,
        example["t_sample"],
        x=example["t_sample"][example["transit_mask_sample"]],
        y=example["lc_sample"][example["transit_mask_sample"]],
        transit_model=example_transit_model,
        transit_parameter=lc_parameter[idx],
        gp_parameter=gp_parameter[idx],
    )
    residuals.append(calculate_whitened_residuals(example["lc_sample"], dist))

In [ ]:
fig, ax = plt.subplots()

ax = plot_residuals(
    ax, example["t_sample"], jax.numpy.array(residuals), credible_region=68
)
ax = plot_masks(ax, example["t_sample"], example["transit_mask_sample"])

ax.legend(loc="center left", bbox_to_anchor=(0, 0.1))
ax.set_ylabel("Whitened Residuals")
ax.set_xlabel("Time")
plt.savefig("../figures/004_plots/residuals_ar_noise.pdf")

In [ ]:
arr = whited_res_samples.T


def bin_and_average(arr, bin_size):
    # Calculate the number of bins
    num_bins = len(arr) // bin_size

    # Reshape the array to (num_bins, bin_size, array_width)
    reshaped_arr = arr[: num_bins * bin_size].reshape(num_bins, bin_size, arr.shape[1])

    # Compute the mean along the bin_size axis
    binned_mean = reshaped_arr.mean(axis=1)

    std = binned_mean.std(axis=0)

    vals = np.percentile(std, [50, 16, 84, 2.5, 95])

    return vals


std_percentiles = []
bin_sizes = range(1, len(arr) // 2 + 1)

# Iterate from bin size 1 to half the length of the array
for bin_size in bin_sizes:
    std_percentiles.append(bin_and_average(arr, bin_size))

vals = np.array(std_percentiles)

In [ ]:
plt.loglog(bin_sizes, vals[:, 0], color=cols[0], label="Mean")
plt.fill_between(
    bin_sizes,
    vals[:, 1],
    vals[:, 2],
    alpha=0.7,
    color=cols[0],
    label=r"68\% credible region",
)
plt.fill_between(
    bin_sizes,
    vals[:, 3],
    vals[:, 4],
    alpha=0.3,
    color=cols[0],
    label=r"95\% credible region",
)

k = [vals[0, 0] / np.sqrt(N) for N in bin_sizes]
plt.plot(bin_sizes, k, color=cols[1], label=r"$1/\sqrt{N}$")

plt.legend(loc="center left", bbox_to_anchor=(0, 0.2))
plt.ylabel("RMS")
plt.xlabel("Bin Size N")

plt.savefig("../figures/004_plots/allandev_ar_noise.pdf")